In [5]:
import geopandas
import numpy as np
from shapely.geometry import LineString

In [6]:
lines = geopandas.read_file("Treefall.shp")
points = []
for geom in lines.geometry:
    if geom.geom_type == "LineString":
        points.extend(list(geom.coords))
    elif geom.geom_type == "MultiLineString":
        for part in geom:
            points.extend(list(part.coords))
points = np.array(points)

In [7]:
centroid = points.mean(axis=0)
centered = points - centroid
cov = np.cov(centered.T)

In [8]:
eigvals, eigvecs = np.linalg.eig(cov)
main_dir = eigvecs[:, np.argmax(eigvals)]
projections = centered @ main_dir
sorted_idx = np.argsort(projections)
sorted_points = points[sorted_idx]
centerline = LineString(sorted_points)
centerline_gdf = geopandas.GeoDataFrame(geometry=[centerline], crs=lines.crs)
centerline_gdf.to_file("tornado_centerline.shp")